In [ ]:
#!/usr/bin/env python3
r"""
Jupyter workflow:

1. Set `playlist_csv_path` to your MiniDisc CSV (INDEX,DURATION,... but empty NAMEs).
2. Set `url` to the Apple Music shared playlist URL.
3. Run.

Output:
- Header row stays the same.
- Row with INDEX == 0:
    NAME column (col 4) becomes the playlist title from Apple (or, if Apple doesn't expose it, we keep whatever was already there).
- Rows with INDEX >= 1:
    NAME column becomes "Title - Artist" for that track index.
- Duration / ENCODING / BITRATE etc. stay untouched.

We parse your CSV using a custom splitter that treats '\,' as a literal comma inside a field
instead of a delimiter.
"""

import sys
import os

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from playlist_sync import csv_utils, scraper


# ---------------------------
# USER VARIABLES (edit these per run)
# ---------------------------

playlist_csv_path = "metal.csv"  # your original CSV
output_csv_path = "output.csv"   # output CSV
url = "https://music.apple.com/us/playlist/your-playlist-name/pl.u-XXXXXXXXXXXXX"  # the Apple Music playlist URL


# ---------------------------
# MAIN EXECUTION
# ---------------------------

# 1. Read your CSV
original_rows = csv_utils.read_csv(playlist_csv_path)

# 2. Pull Apple Music page
playlist_name, track_displays = scraper.fetch_playlist(url)

# 3. Inject names back into your CSV rows
merged_rows, warnings = csv_utils.inject_names_by_index(
    original_rows,
    playlist_name,
    track_displays
)

# 4. Print warnings
for warning in warnings:
    print(f"Warning: {warning}")

# 5. Write output
csv_utils.write_csv(output_csv_path, merged_rows)

# 6. Display results
print(f"Playlist: {playlist_name}")
print(f"Tracks: {len(track_displays)}")
print(f"Output: {output_csv_path}")
print()
for row in merged_rows:  # show first 5 tracks
    print(csv_utils.join_preserving_escaped_commas(row))